<a href="https://colab.research.google.com/github/grmntfrancis0/earthengine-community/blob/main/Temporal_interpolation_colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install --upgrade xee

In [ ]:
!pip install -U geemap

In [ ]:
!pip install rioxarray

In [ ]:
!pip install geopandas

In [ ]:
!pip install -U seaborn

In [ ]:
import rioxarray
import ee
import geemap
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import xarray as xr
import xee
import geopandas as gpd

In [ ]:
import ee

In [ ]:
ee.Authenticate()
ee.Initialize(project = "ee-grmntfrancis0",
              opt_url = 'https://earthengine-highvolume.googleapis.com')

In [ ]:
import geemap

In [ ]:
map = geemap.Map()
map

In [ ]:
roi = map.draw_last_feature.geometry()
roi

In [ ]:
border = border = (
    ee.FeatureCollection("WM/geoLab/geoBoundaries/600/ADM2")
    .filterBounds(roi)
)

In [ ]:
map.addLayer(border)

In [ ]:
vec = geemap.ee_to_gdf(border)
vec

In [ ]:
temp = (
    ee.ImageCollection("NASA/VIIRS/002/VNP21A1D")
    .filterDate("2020", "2021")
    .select("LST_1KM")
)
temp

In [ ]:
ndvi = (
    ee.ImageCollection("NASA/VIIRS/002/VNP13A1")
    .filterDate("2020", "2021")
    .select("NDVI")
)
ndvi

In [ ]:
import xarray as xr

In [ ]:
ds_temp = xr.open_dataset(
    temp,
    engine = "ee",
    crs = "EPSG:4326",
    geometry = border.geometry(),
    scale = 0.01
)

In [ ]:
df = ds_temp.to_dataframe().dropna()

In [ ]:
ds_ndvi = xr.open_dataset(
    ndvi,
    engine = "ee",
    crs = "EPSG:4326",
    geometry = border.geometry(),
    scale = 0.001
)

In [ ]:
df = ds_ndvi.to_dataframe().dropna()

In [ ]:
ds_temp = ds_temp.sortby("time") * 1
ds_ndvi = ds_ndvi.sortby("time") * 1

In [ ]:
ds_temp_monthly = ds_temp.resample(time = "M").mean("time")
ds_ndvi_monthly = ds_ndvi.resample(time = "M").median("time")

In [ ]:
ds_ndvi_like_temp = ds_ndvi_monthly.interp_like(ds_temp_monthly)

In [ ]:
ds_temp_monthly
ds_ndvi_like_temp

In [ ]:
merge = xr.merge([ds_temp_monthly, ds_ndvi_like_temp])
merge

In [ ]:
df_clipped = ds_ndvi.to_dataframe().dropna()

In [ ]:
df_clipped = ds_temp.to_dataframe().dropna()

In [ ]:
!pip install netCDF4

In [ ]:
import netCDF4 as nc

In [ ]:
merge.to_netcdf("merge_ds.nc")

In [ ]:
import matplotlib.pyplot as plt

In [ ]:
import geopandas as gpd

In [ ]:
vec.plot()

In [ ]:
vec.crs

In [ ]:
vec.geometry

In [ ]:
ds_rename = merge.rename({"lon": "x", "lat": "y"})

In [ ]:
ds_clip = ds_rename.rio.clip(vec.geometry, vec.crs)

In [ ]:
import pandas as pd

In [ ]:
ds_clip.LST_1KM.plot.contourf(
    x = "x",
    y = "y",
    col = "time",
    col_wrap = 4,
    robust = True,
    levels = 10,
    title = "Temporal interpolation (2020-2021)", fontsize = 18,
    cbar_kwargs ={"orientation":"vertical", "label":"temporal interpolation", "shrink":0.9, "pad":0.07}
)
plt.savefig('temp_lainya', dpi = 360, bbox_inches = 'tight')

In [ ]:
ds_clip.LST_1KM.plot.contourf(
    x = "x", y = "y", col = "time", col_wrap = 4, cmap = "gist_rainbow_r", robust = True, levels = 10,
    title = "Clipped LST_1KM (2020-2021)"
)
plt.savefig('temp_lainya1', dpi = 360, bbox_inches = 'tight')

In [ ]:
ds_clip.NDVI.plot.contourf(
    x = "x", y = "y", col = "time", col_wrap = 4, cmap = 'gist_earth_r', robust = True, levels = 10,
    title = "Clipped NDVI (2020-2021)"
)
plt.savefig('ndvi_lainya', dpi = 360, bbox_inches = 'tight')